# WSMTE — 04_model_training.ipynb
**KAGGLE GPU (T4 x2)** | Day 4 Tasks 4.1–4.14

Trains Configs A–G (10 seeds each). Saves results after every config.

**Kaggle input dataset required:** `wsmte-processed-data`
- Path: `/kaggle/input/wsmte-processed-data/`
- Contains: X_train/val/test.npy, y_clf/reg_*.npy, class_weights.json

In [ ]:
# ── Cell 1: Clone repo + sys.path + CONFIG ─────────────────────────────────
!git clone https://github.com/IAMNEERAJ05/WSMTE.git
import sys
sys.path.insert(0, '/kaggle/working/WSMTE')

from config.config import CONFIG
print(f"Window size    : {CONFIG['window_size']}")
print(f"n_features     : {CONFIG['n_features']}")
print(f"Batch size     : {CONFIG['batch_size']}")
print(f"Max epochs     : {CONFIG['max_epochs']}")
print(f"ES patience    : {CONFIG['early_stopping_patience']}")

## GPU Check

In [ ]:
# ── Cell 2: Verify GPU ────────────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'TF version : {tf.__version__}')
print(f'GPUs found : {len(gpus)}')
for g in gpus:
    print(f'  {g}')
if not gpus:
    print('WARNING: No GPU — training will be very slow')

## Load All Data Arrays

In [ ]:
# ── Cell 3: Load processed data from Kaggle dataset ───────────────────────
import numpy as np
import json
import pandas as pd

DATA_DIR = '/kaggle/input/wsmte-processed-data/'

X_train = np.load(DATA_DIR + 'X_train.npy')
X_val   = np.load(DATA_DIR + 'X_val.npy')
X_test  = np.load(DATA_DIR + 'X_test.npy')

y_clf_train = np.load(DATA_DIR + 'y_clf_train.npy')
y_clf_val   = np.load(DATA_DIR + 'y_clf_val.npy')
y_clf_test  = np.load(DATA_DIR + 'y_clf_test.npy')

y_reg_train = np.load(DATA_DIR + 'y_reg_train.npy')
y_reg_val   = np.load(DATA_DIR + 'y_reg_val.npy')
y_reg_test  = np.load(DATA_DIR + 'y_reg_test.npy')

with open(DATA_DIR + 'class_weights.json') as f:
    class_weight = json.load(f)

print(f'X_train : {X_train.shape}  dtype={X_train.dtype}')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_clf_train : {y_clf_train.shape}  labels={set(y_clf_train.tolist())}')
print(f'y_reg_train : {y_reg_train.shape}')
print(f'class_weight: {class_weight}')

data = {
    'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
    'y_clf_train': y_clf_train, 'y_clf_val': y_clf_val, 'y_clf_test': y_clf_test,
    'y_reg_train': y_reg_train, 'y_reg_val': y_reg_val, 'y_reg_test': y_reg_test,
    'class_weight': class_weight,  # key must match data.get('class_weight') in trainer
}
print('data dict ready OK')

## Training Loop — Configs A–G
> Each config: 10 seeds. Results saved to CSV after every config.
> If the session dies, check `ablation_results_partial.csv` and restart from the last incomplete config.

In [ ]:
# ── Cell 4: Import trainer ────────────────────────────────────────────────
from src.training.trainer import train_multi_run

RESULTS_PATH = '/kaggle/working/ablation_results_partial.csv'
print('Trainer imported OK')
print(f'Results will be saved to: {RESULTS_PATH}')

In [ ]:
# ── Cell 5: Run Config A (technicals only, classification) ────────────────
cfg_a = CONFIG['ablation_configs']['A']
print(f"Config A: {cfg_a['description']}")
print(f"Features ({len(cfg_a['features'])}): {cfg_a['features']}")

results_A = train_multi_run(
    ablation_cfg=cfg_a,
    config_name='A',
    data=data,
    config=CONFIG,
    n_runs=cfg_a['n_runs'],
    results_path=RESULTS_PATH,
)
print(f'Config A done. Mean accuracy: {results_A["accuracy"].mean():.4f} ')
print(results_A[['run','accuracy','auc','f1']].to_string(index=False))

In [ ]:
# ── Cell 6: Run Config B (technicals + company sentiment) ─────────────────
cfg_b = CONFIG['ablation_configs']['B']
print(f"Config B: {cfg_b['description']}")

results_B = train_multi_run(
    ablation_cfg=cfg_b,
    config_name='B',
    data=data,
    config=CONFIG,
    n_runs=cfg_b['n_runs'],
    results_path=RESULTS_PATH,
)
print(f'Config B done. Mean accuracy: {results_B["accuracy"].mean():.4f}')
print(results_B[['run','accuracy','auc','f1']].to_string(index=False))

In [ ]:
# ── Cell 7: Run Config C (all 9 features, classification) ─────────────────
cfg_c = CONFIG['ablation_configs']['C']
print(f"Config C: {cfg_c['description']}")

results_C = train_multi_run(
    ablation_cfg=cfg_c,
    config_name='C',
    data=data,
    config=CONFIG,
    n_runs=cfg_c['n_runs'],
    results_path=RESULTS_PATH,
)
print(f'Config C done. Mean accuracy: {results_C["accuracy"].mean():.4f}')

In [ ]:
# ── Cell 8: Run Config D (full features, single-task confirmation) ─────────
cfg_d = CONFIG['ablation_configs']['D']
print(f"Config D: {cfg_d['description']}")

results_D = train_multi_run(
    ablation_cfg=cfg_d,
    config_name='D',
    data=data,
    config=CONFIG,
    n_runs=cfg_d['n_runs'],
    results_path=RESULTS_PATH,
)
print(f'Config D done. Mean accuracy: {results_D["accuracy"].mean():.4f}')

In [ ]:
# ── Cell 9: Run Config E (all 9 features, classification only — MTL ablation) 
cfg_e = CONFIG['ablation_configs']['E']
print(f"Config E: {cfg_e['description']}")

results_E = train_multi_run(
    ablation_cfg=cfg_e,
    config_name='E',
    data=data,
    config=CONFIG,
    n_runs=cfg_e['n_runs'],
    results_path=RESULTS_PATH,
)
print(f'Config E done. Mean accuracy: {results_E["accuracy"].mean():.4f}')

In [ ]:
# ── Cell 10: Run Config F (regression only — MTL ablation) ───────────────
cfg_f = CONFIG['ablation_configs']['F']
print(f"Config F: {cfg_f['description']}")

results_F = train_multi_run(
    ablation_cfg=cfg_f,
    config_name='F',
    data=data,
    config=CONFIG,
    n_runs=cfg_f['n_runs'],
    results_path=RESULTS_PATH,
)
print(f'Config F done. Mean RMSE: {results_F["rmse"].mean():.6f}')

In [ ]:
# ── Cell 11: Run Config G (both heads, concat — full WSMTE without PSO) ──
cfg_g = CONFIG['ablation_configs']['G']
print(f"Config G: {cfg_g['description']}")

results_G = train_multi_run(
    ablation_cfg=cfg_g,
    config_name='G',
    data=data,
    config=CONFIG,
    n_runs=cfg_g['n_runs'],
    results_path=RESULTS_PATH,
)
print(f'Config G done. Mean accuracy: {results_G["accuracy"].mean():.4f}')
print(results_G[['run','accuracy','balanced_accuracy','auc','f1','rmse']].to_string(index=False))

## Save Config G Best Model Weights

In [ ]:
# ── Cell 12: Train Config G one more time to get best model for PSO stage ─
# We need the actual Keras model object for Stage 2 PSO search.
# Train with seed=0 (or the seed that gave best accuracy in results_G).
from src.models.wsmte import build_wsmte
from src.training.callbacks import get_callbacks

best_seed_g = int(results_G.loc[results_G['accuracy'].idxmax(), 'seed'])
print(f'Best Config G seed: {best_seed_g}  '
      f'accuracy={results_G["accuracy"].max():.4f}')

tf.random.set_seed(best_seed_g)
np.random.seed(best_seed_g)

feat_idx = [CONFIG['feature_columns'].index(f) for f in cfg_g['features']]
X_tr = X_train[:, :, feat_idx]
X_vl = X_val[:, :, feat_idx]

g_model = build_wsmte(CONFIG, ablation_cfg=cfg_g)
g_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['learning_rate'])
)
g_model.fit(
    X_tr, [y_reg_train, y_clf_train],
    validation_data=(X_vl, [y_reg_val, y_clf_val]),
    epochs=CONFIG['max_epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=get_callbacks(CONFIG, run_id='config_G_best'),
    verbose=1,
)
print('Config G best model trained OK')

In [ ]:
# ── Cell 13: Save Config G weights ────────────────────────────────────────
g_model.save_weights('/kaggle/working/config_g_best.weights.h5')
print('Saved -> /kaggle/working/config_g_best.weights.h5')

# Also save full model for portability
g_model.save('/kaggle/working/config_g_best.keras')
print('Saved -> /kaggle/working/config_g_best.keras')

## Save All Results

In [ ]:
# ── Cell 14: Combine and save final results CSV ───────────────────────────
all_results = pd.concat(
    [results_A, results_B, results_C, results_D,
     results_E, results_F, results_G],
    ignore_index=True
)
all_results.to_csv('/kaggle/working/ablation_results_AG.csv', index=False)
print(f'Saved -> /kaggle/working/ablation_results_AG.csv')
print(f'Shape : {all_results.shape}')
print()
summary = all_results.groupby('config')['accuracy'].agg(['mean','max','std'])
summary.columns = ['mean_acc', 'max_acc', 'std_acc']
print(summary.round(4).to_string())

## Download Instructions

In [ ]:
# ── Cell 15: Remind what to download ──────────────────────────────────────
import os
downloads = [
    '/kaggle/working/ablation_results_AG.csv',
    '/kaggle/working/config_g_best.weights.h5',
    '/kaggle/working/config_g_best.keras',
]
print('Files to download from Kaggle Output:')
for p in downloads:
    size = os.path.getsize(p) // 1024 if os.path.exists(p) else 'MISSING'
    print(f'  {p}  ({size} KB)')
print()
print('Local destinations:')
print('  ablation_results_AG.csv  -> ablation/')
print('  config_g_best.*          -> results/saved_models/')
print('Expected Kaggle runtime: 2-4 hours')